# Inferencia regional de severidad GRD 2024

Este notebook implementa intervalos de confianza y pruebas de hipótesis sobre diferencias regionales de severidad.

Salida principal:
- IC95 para proporción de casos graves por región.
- Comparaciones pareadas entre regiones (z-test de proporciones).
- Prueba global entre regiones (ANOVA/Kruskal-Wallis sobre tasa por comuna).

La siguiente celda carga las bases procesadas y deja listas las variables necesarias para el análisis regional.

In [1]:
from pathlib import Path
from math import sqrt
from statistics import NormalDist

import pandas as pd
from IPython.display import display

ROOT_DIR = Path.cwd().resolve()
for candidate in [ROOT_DIR, *ROOT_DIR.parents]:
    if (candidate / 'app').exists() and (candidate / 'data').exists():
        ROOT_DIR = candidate
        break

processed_dir = ROOT_DIR / 'data' / 'processed'
resumen_region = pd.read_csv(processed_dir / 'severidad_region.csv')
resumen_comuna = pd.read_csv(processed_dir / 'severidad_comuna.csv')

print(f'Raiz del proyecto: {ROOT_DIR}')
print('Regiones cargadas:', len(resumen_region))
print('Comunas cargadas:', len(resumen_comuna))

Raiz del proyecto: C:\Users\Bato\Desktop\Infe\Proyecto-ADIE
Regiones cargadas: 16
Comunas cargadas: 333


La siguiente celda resume la severidad por región y calcula las proporciones base para el análisis.

In [2]:
def wilson_ci(successes: float, n: float, confidence: float = 0.95):
    """IC de Wilson para una proporcion."""
    if n <= 0:
        return (float('nan'), float('nan'))
    z = NormalDist().inv_cdf(1 - (1 - confidence) / 2)
    p = successes / n
    denom = 1 + (z**2) / n
    center = (p + (z**2) / (2 * n)) / denom
    margin = (z / denom) * sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))
    return (max(0.0, center - margin), min(1.0, center + margin))


def two_proportion_z_test(x1: float, n1: float, x2: float, n2: float):
    """z-test bilateral para comparar dos proporciones."""
    if n1 <= 0 or n2 <= 0:
        return float('nan'), float('nan')

    p_pool = (x1 + x2) / (n1 + n2)
    se = sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    if se == 0:
        return float('nan'), float('nan')

    z = (x1 / n1 - x2 / n2) / se
    p_value = 2 * (1 - NormalDist().cdf(abs(z)))
    return z, p_value

Aquí se construyen los intervalos de confianza por región usando la aproximación de Wilson.

In [3]:
tabla_ic_region = resumen_region.copy()
tabla_ic_region['total'] = pd.to_numeric(tabla_ic_region['total'], errors='coerce')
tabla_ic_region['alta'] = pd.to_numeric(tabla_ic_region['alta'], errors='coerce')

ci_low = []
ci_high = []
for _, row in tabla_ic_region.iterrows():
    low, high = wilson_ci(row['alta'], row['total'], confidence=0.95)
    ci_low.append(low * 100)
    ci_high.append(high * 100)

tabla_ic_region['prop_graves'] = (tabla_ic_region['alta'] / tabla_ic_region['total']) * 100
tabla_ic_region['ic95_inf'] = ci_low
tabla_ic_region['ic95_sup'] = ci_high

tabla_ic_region = tabla_ic_region.sort_values('prop_graves', ascending=False)

print('Intervalos de confianza (95%) por region:')
display(tabla_ic_region[['REGION', 'total', 'alta', 'prop_graves', 'ic95_inf', 'ic95_sup']])

Intervalos de confianza (95%) por region:


,REGION,total,alta,prop_graves,ic95_inf,ic95_sup
11,METROPOLITANA,339425,77321,22.779996,22.639207,22.921400
14,VALPARAISO,94595,20567,21.742164,21.480451,22.006172
12,O’HIGGINS,44692,9125,20.417524,20.046357,20.793777
4,BIOBIO,117156,23450,20.016047,19.787915,20.246145
9,MAGALLANES,13427,2668,19.870410,19.204140,20.553916
7,LOS LAGOS,58867,11294,19.185622,18.869551,19.505714
15,ÑUBLE,13714,2532,18.462885,17.822375,19.121057
10,MAULE,81911,14409,17.591044,17.331824,17.853303
8,LOS RIOS,23547,4013,17.042511,16.567637,17.528136
6,LA ARAUCANIA,79424,13142,16.546636,16.289821,16.806686


La siguiente celda compara regiones por pares con una prueba de dos proporciones.

In [4]:
# Comparaciones pareadas entre regiones (z-test de proporciones)
comparaciones = []
rows = tabla_ic_region.reset_index(drop=True)

for i in range(len(rows)):
    for j in range(i + 1, len(rows)):
        r1 = rows.iloc[i]
        r2 = rows.iloc[j]
        z, p = two_proportion_z_test(r1['alta'], r1['total'], r2['alta'], r2['total'])
        comparaciones.append({
            'region_1': r1['REGION'],
            'region_2': r2['REGION'],
            'z_score': z,
            'p_value': p,
            'diferencia_pp': (r1['alta'] / r1['total'] - r2['alta'] / r2['total']) * 100,
        })

comparaciones_df = pd.DataFrame(comparaciones).sort_values('p_value', ascending=True)

print('Top comparaciones con menor p-value (sin ajuste multiple):')
display(comparaciones_df.head(15))

Top comparaciones con menor p-value (sin ajuste multiple):


,region_1,region_2,z_score,p_value,diferencia_pp
1,METROPOLITANA,O’HIGGINS,11.242021,0.0,2.362471
2,METROPOLITANA,BIOBIO,19.668272,0.0,2.763949
5,METROPOLITANA,ÑUBLE,11.848564,0.0,4.317111
4,METROPOLITANA,LOS LAGOS,19.356365,0.0,3.594374
6,METROPOLITANA,MAULE,32.298599,0.0,5.188952
7,METROPOLITANA,LOS RIOS,20.418138,0.0,5.737485
9,METROPOLITANA,ANTOFAGASTA,27.747376,0.0,6.595248
8,METROPOLITANA,LA ARAUCANIA,38.430047,0.0,6.233360
12,METROPOLITANA,ATACAMA,27.597431,0.0,8.264260
13,METROPOLITANA,ARICA Y PARINACOTA,33.077287,0.0,9.972212


Por último, se aplican pruebas globales para evaluar si existen diferencias estadísticamente significativas entre regiones.

In [5]:
# Prueba global entre regiones sobre tasa comunal de severidad
try:
    from scipy import stats
except ImportError:
    stats = None

comuna_tmp = resumen_comuna.copy()
comuna_tmp['total'] = pd.to_numeric(comuna_tmp['total'], errors='coerce')
comuna_tmp['alta'] = pd.to_numeric(comuna_tmp['alta'], errors='coerce')
comuna_tmp = comuna_tmp[comuna_tmp['total'] > 0].copy()
comuna_tmp['prop_graves_comuna'] = comuna_tmp['alta'] / comuna_tmp['total']

groups = [
    g['prop_graves_comuna'].dropna().values
    for _, g in comuna_tmp.groupby('REGION_GEOJSON')
    if len(g) >= 3
]

if stats is None:
    print('scipy no instalado: no se puede correr ANOVA/Kruskal en este notebook.')
elif len(groups) < 2:
    print('No hay suficientes grupos para prueba global.')
else:
    f_stat, p_anova = stats.f_oneway(*groups)
    h_stat, p_kruskal = stats.kruskal(*groups)
    print('ANOVA (tasa comunal de severidad entre regiones):')
    print(f'  F = {f_stat:.4f}, p-value = {p_anova:.6g}')
    print('Kruskal-Wallis (alternativa no parametrica):')
    print(f'  H = {h_stat:.4f}, p-value = {p_kruskal:.6g}')

ANOVA (tasa comunal de severidad entre regiones):
  F = 3.8603, p-value = 2.38227e-06
Kruskal-Wallis (alternativa no parametrica):
  H = 108.7588, p-value = 2.79014e-16
